# Wireless Coverage — Part 3: H3 Gridding of DEM / DSM / CHM Surfaces

Prepare the LiDAR-derived elevation surfaces from Part 2 for global-grid analysis by
H3-indexing each surface. DEM and DSM elevation bands are extracted as contiguous isoband
polygons (`rst_isoband`) and indexed to H3 cells with the Databricks product
`h3_try_coverash3`. The Canopy Height Model is aggregated to the maximum canopy height per
H3 cell (`gbx_rst_h3_rastertogridmax`). Gaps over land are filled with `h3_cellfill`
(k=1, IDW). The resulting H3 tables are prepared for downstream coverage analysis — antenna
siting and line-of-sight in Part 4.

> **Runtime.** Runs on **Databricks Serverless environment 5** — the lightweight tier (pure
> Python/PySpark, no JAR or GDAL init script). Set `DEMO=True` (default) for a Golden Gate
> Park sub-AOI (~25 tiles) to iterate quickly; `DEMO=False` for full San Francisco.

---
**Last Update:** 2026-10-01


In [ ]:
%pip install --quiet --disable-pip-version-check --no-deps --force-reinstall \
  "geobrix[light_env5,vizx,overture] @ file:///Volumes/geospatial_docs/geobrix/sample-data/geobrix-0.5.2-py3-none-any.whl"
%pip install --quiet \
  "geobrix[light_env5,vizx,overture] @ file:///Volumes/geospatial_docs/geobrix/sample-data/geobrix-0.5.2-py3-none-any.whl"


In [ ]:
%restart_python


In [ ]:
import warnings, glob, math
warnings.filterwarnings("ignore")
from pyspark.sql import functions as F
from pyspark.databricks.sql import functions as DBF
from shapely import wkb as _wkb
from shapely.geometry import box as _box
from shapely.ops import unary_union

from databricks.labs.gbx.pyrx import functions as rx
from databricks.labs.gbx.pygx import functions as gx
from databricks.labs.gbx.pygx import _cellfill
from databricks.labs.gbx.ds.register import register
from databricks.labs.gbx.vizx import (
    plot_static, plot_point_cloud,
    raster_layer, vector_layer, point_cloud_layer, cells_as_gdf,
)
from databricks.labs.gbx.sample.overture import OvertureClient

rx.register(spark)
gx.register(spark)
register(spark)


In [ ]:
# --- source surfaces (produced by nb2) ---
CATALOG      = "geospatial_docs"
SCHEMA       = "geobrix"
SRC_PREFIX   = "wc_surface"            # wc_surface_{dsm,dtm,chm}
OUT_PREFIX   = "wc_h3"                 # wc_h3_{dem,dsm,chm}
LAZ_DIR      = "/Volumes/geospatial_docs/wireless_coverage/data/lidar/sf/laz"
LAND_DIR     = "/Volumes/geospatial_docs/geobrix/sample-data/geobrix-examples/sf/overture-water-bay"

# --- H3 grid ---
H3_RESOLUTION = 9                      # ≈174 m edge; set 8 (≈460 m) for a fast coarse pass
# 10 elevation bands × 30 m — matches SF DTM/DSM range (~0–280 m).
BREAKS_M     = [float(b) for b in range(0, 301, 30)]

# --- scope: DEMO=True limits to Golden Gate Park ±2 km tiles (~25 tiles) ---
DEMO         = True
SF_AOI       = (-122.55, 37.70, -122.35, 37.85)   # full San Francisco lon/lat BBOX
TILE_M       = 1024.0                  # matches nb1/nb2 tile size

FORCE_RELOAD = False                   # True → rebuild H3 tables; False → reuse


In [ ]:
def _persist(df, name):
    """Write to ``CATALOG.SCHEMA.OUT_PREFIX_<name>`` and return it read back.
    FORCE_RELOAD rebuilds; otherwise an existing table is reused."""
    tbl = f"{CATALOG}.{SCHEMA}.{OUT_PREFIX}_{name}"
    if FORCE_RELOAD or not spark.catalog.tableExists(tbl):
        df.write.mode("overwrite").saveAsTable(tbl)
    return spark.table(tbl)


In [ ]:
dsm = spark.table(f"{CATALOG}.{SCHEMA}.{SRC_PREFIX}_dsm")
dtm = spark.table(f"{CATALOG}.{SCHEMA}.{SRC_PREFIX}_dtm")
chm = spark.table(f"{CATALOG}.{SCHEMA}.{SRC_PREFIX}_chm")
print(f"Source surfaces: DSM {dsm.count()} | DTM {dtm.count()} | CHM {chm.count()} tiles")

# Confirm CRS (all should be EPSG:3857 — requires rst_transform before H3 indexing).
sample_srid = dsm.select(F.expr("gbx_rst_srid(dsm)").alias("srid")).first()["srid"]
assert sample_srid == 3857, f"Expected EPSG:3857, got {sample_srid}"
print(f"Surface CRS confirmed: EPSG:{sample_srid}")


In [ ]:
laz_files = sorted(glob.glob(f"{LAZ_DIR}/**/*.laz", recursive=True))
if laz_files:
    plot_point_cloud(
        laz_files[0], color="rgb", max_points=150_000,
        title="SF LiDAR — raw returns (one EPT node, RGB-colored)",
    )
else:
    print(f"No .laz staged under {LAZ_DIR}; point cloud preview skipped.")


In [ ]:
from databricks.labs.gbx.sample.lidar import lonlat_to_3857, aoi_lonlat_to_3857
X0, Y0, X1, Y1 = aoi_lonlat_to_3857(*SF_AOI)

if DEMO:
    _gx, _gy = lonlat_to_3857(-122.486, 37.769)   # Golden Gate Park centroid
    _tx, _ty = int((_gx - X0) / TILE_M), int((_gy - Y0) / TILE_M)
    DEMO_FILTER = (
        F.col("tx").between(_tx - 2, _tx + 2) &
        F.col("ty").between(_ty - 2, _ty + 2)
    )
    dtm_in = dtm.where(DEMO_FILTER)
    dsm_in = dsm.where(DEMO_FILTER)
    chm_in = chm.where(DEMO_FILTER)
    print(f"DEMO mode: {dtm_in.count()} DTM | {dsm_in.count()} DSM | {chm_in.count()} CHM tiles "
          f"in ±2 km window around Golden Gate Park (tx={_tx}, ty={_ty})")
else:
    dtm_in, dsm_in, chm_in = dtm, dsm, chm
